# <span style="color:blue;font-weight:bold;font-size:1.1em">Day 16: Matched Pairs $t$-Test</span>

We have three different versions of the $t$-test that we can use:

- $t$-Test (1 Sample)
- Independent Samples $t$-Test
- Matched Pairs $t$-Test

Today, we learn about the **Matched Pairs** $\textbf{t}$**-test** which is actually a innovative use of the orginal $t$-Test where the the single sample used to run the $t$-test is a difference score calculated using the two samples of data provided. The quintessential use for the Matched Pairs $t$-Test is a
<center><span style="color:blue;font-weight:bold;font-size:1.1em">Pretest vs. Posttest</span></center>

research design.

In [22]:
from datascience import *
%matplotlib inline
import matplotlib.pyplot as plots
plots.style.use('fivethirtyeight')
import numpy as np
import seaborn as sns
import scipy.stats as stats
from scipy.stats import gaussian_kde

In [23]:
def describe(data, var = None):
  if isinstance(data, Table):
    data = data.column(var)
  names = np.array(["mean", "std", "n", " : : ", "min", "Q1", "median", "Q3", "max", "IQR"])
  values = np.array([round(np.mean(data),1), round(stats.tstd(data),2), len(data) - 1, "", min(data), np.percentile(data, 25), np.median(data), np.percentile(data, 75), max(data), stats.iqr(data)])
  tab = Table().with_columns('Names', names, 'Values', values)  ;  transposed_tab = Table().with_columns( zip(tab.column('Names'), tab.column('Values')) ).show()
  return

def describes(data, num, group):
  dat = data.column(num)  ;  group_var = data.column(group)  ;  groups = np.unique(group_var)  ;  rows = []  ;  tab = Table(["mean", "std", "n", " : : ", "min", "Q1", "median", "Q3", "max", "IQR"])
  for g in groups:
    # Filter data for the current group
    sub_dat = dat[group_var == g]  ;    rows = [g, round(np.mean(sub_dat), 1), round(stats.tstd(sub_dat), 2), len(sub_dat) - 1, min(sub_dat), np.percentile(sub_dat, 25), np.median(sub_dat), np.percentile(sub_dat, 75), max(sub_dat), stats.iqr(sub_dat)]        
    tab = tab.with_row(rows)
  tab.show()
  return 

def box(data, num = None, sm = False):
  if isinstance(data, Table):
    df = data.to_df()  ;  sns.boxplot(data=df, y=num, width=0.45, showmeans=sm)  ;
    plots.title(f'Boxplot of {num}')  ;  plots.xticks([], [])  ;  plots.show()  
  else:
    if num == True:
      sm = True  ;  num = 'Variable'
    if num == None:
      num = 'Variable'
    plots.figure(figsize=(6, 6))  ;  plots.boxplot(data, widths = 0.45, showmeans=sm)
    plots.title(f'Boxplot of {num}')  ;  plots.xticks([], [])  ;  plots.show()
  return

def boxes(data, num, cat, sm = False):
  if not isinstance(data, Table):
    print("ERROR: First input must be table name.")
  else:
    df = data.to_df()  ;  sns.boxplot(data=data, x=cat, y=num, width=0.75, showmeans=sm)
    plots.title(f'Boxplot of {num} Grouped by {cat}')  ;  plots.show()
  return

def density(data, num = None):
  if isinstance(data, Table):
    sns.kdeplot( data = data.column(num), bw_method="scott", bw_adjust=0.75, cut=3 )  ;  plots.title(f'Density Plot of {num}')  ;  plots.show()
  else:
    if num == None:
      num = 'Variable'  ;  sns.kdeplot( data = data, bw_method="scott", bw_adjust=0.75, cut=3 )  ;  plots.title(f'Density Plot of {num}')  ;  plots.show()
    else:
      sns.kdeplot( data = data, bw_method="scott", bw_adjust=0.75, cut=3 )  ;  plots.title(f'Density Plot of {num}')  ;  plots.show()
  return

def densities(tab, var, group_var):
  groups = tab.column(group_var) ;  unique_groups = np.unique(groups)  ;  plots.figure()
  for g in unique_groups:
    # Filter data for the current group
    mask = (groups == g)
    x = tab.column(var)[mask]   
    # Calculate KDE for the group subset
    kde = gaussian_kde(x)
    x_grid = np.linspace(x.min() - 1, x.max() + 1, 512)  ;    plots.plot(x_grid, kde(x_grid), label=str(g))   
  plots.title(f'Density Plot of {var} by {group_var}')  ;  plots.xlabel(f'{var}')  ;  plots.ylabel('Density')
  plots.legend()  ;  plots.show()
  return

### Data Checks for $t$-Tests

The data checks are identical to the 1 Sample $t$-test case because we are running a $t$-test on the difference scores (sometimes called the **gain** scores).

<table style="width:75%">
  <tr>
    <th>Description</th>
    <th>Sample Size</th> 
    <th>Data Checks</th> 
    <th>Graphics</th>
    
  </tr>
  <tr>
    <td style="text-align: center; vertical-align: middle;">Tiny</td>
    <td style="text-align: center; vertical-align: middle;">$n\leq 15$</td>
    <td style="text-align: center; vertical-align: middle;">We need evidence the sample was drawn from a <span style="font-weight:bold">bell-shaped</span> distribution, and <span style="font-weight:bold">no outliers</spac>.</td>
    <td style="text-align: center; vertical-align: middle;">Check histogram or density plot (bell-shape), and check box plot (no outliers).</td>
  </tr>
  <tr>
    <td style="text-align: center; vertical-align: middle;">Small</td>
    <td style="text-align: center; vertical-align: middle;">$15<n < 40$</td>
    <td style="text-align: center; vertical-align: middle;">We <span style="font-weight:bold">prefer</span> evidence the sample was drawn from a bell-shaped distribution, and limited (or balanced) outliers.</td>
    <td style="text-align: center; vertical-align: middle;">Check histogram or density plot (bell-shape preferred), check box plot (limited outliers)</td>
  </tr>
  <tr>
    <td style="text-align: center; vertical-align: middle;">Large</td>
    <td style="text-align: center; vertical-align: middle;">$n\geq 40$</td>
    <td style="text-align: center; vertical-align: middle;">We prefer evidence the sample was drawn from a bell-shaped distribution and note that massive outliers and skew may erode the accuracy of results (p-values).</td>
    <td style="text-align: center; vertical-align: middle;">No data checks required due to the <span style="font-weight:bold">robustness of the t-statistic</span>.</td>
  </tr>
</table>

## <span style="color:blue;font-weight:bold;font-size:1.1em">Example 1: Cybersecurity Training Effectiveness</span>

**The Situation:** A logistics company measures the number of weekly phishing simulation links clicked by 12 specific employees over a 3-month baseline period. The company then deploys an updated cybersecurity training program and tracks the same 12 employees' error counts over the next 3 months.

**Research Question:** Test at the $\alpha = 0.01$ level whether the cybersecurity training reduces the mean number of employee phishing errors (given the results data shown below).

In [24]:
Baseline_Errors = make_array(14,8,19,11,6,15,22,9,12,17,5,14)
Post_Training_Errors = make_array(9,6,13,11,4,10,16,7,8,11,5,9)
cyber_train = Table().with_columns('Baseline Errors', Baseline_Errors, 'Post-Training Errors', Post_Training_Errors)
cyber_train

Baseline Errors,Post-Training Errors
14,9
8,6
19,13
11,11
6,4
15,10
22,16
9,7
12,8
17,11


### Hypothesis

### Data Checks


### Run Test

### Reporting Out

## <span style="color:blue;font-weight:bold;font-size:1.1em">Example 2: Sleep Deprivation</span>

Does acute **sleep deprivation affect cognitive processing speed**? To address this question, 15 neurobiology students completed a standardized computerized reaction-time task (measured in milliseconds) twice: once after a full 8-hour night of sleep, and once after being awake for 24 hours. The order of conditions was randomized to eliminate learning effects.

Test at the $\alpha = 0.05$ level whether sleep deprivation resulted in a loss of cognitive processing speed using the data shown below.

In [25]:
eight_hrs = make_array(245, 230, 260, 280, 225, 210, 255, 240, 270, 235, 220, 248, 265, 250, 232)
awake_24 = make_array(280, 242, 295, 275, 260, 250, 280, 245, 310, 265, 230, 290, 272, 285, 261)
sleep = Table().with_columns('8 Hours of Sleep', eight_hrs, 'Awake 24 Hours', awake_24)
sleep

8 Hours of Sleep,Awake 24 Hours
245,280
230,242
260,295
280,275
225,260
210,250
255,280
240,245
270,310
235,265


### Hypothesis

### Data Checks

### Run Test

### Reporting Out

## <span style="color:blue;font-weight:bold;font-size:1.1em">Example 3: Biomechanics</span>

**Research Question:** Does a new minimalist carbon-fiber running shoe decrease the oxygen consumption of long-distance runners compared to standard training shoes?

**Study Design:** 20 competitive collegiate runners completed a 15-minute treadmill test at a constant submaximal speed $12$ km/h under two conditions: wearing their standard training shoes and wearing the carbon-fiber shoes. The $\text{VO}_{2}$ was measured in $\text{mL/kg/min}$.

In [26]:
std_shoe = make_array(54.2, 51.5, 55.0, 53.8, 52.1, 56.4, 50.9, 52.7, 54.5, 53.0, 55.2, 51.8, 52.4, 56.1, 53.9, 54.7, 50.5, 52.9, 55.6, 53.3)
carbon_fiber = make_array(52.1, 50.3, 54.8, 51.2, 51.9, 53.5, 49.1, 52.4, 52.0, 51.1, 53.4, 50.2, 52.6, 53.9, 51.5, 53.0, 48.9, 51.0, 53.1, 51.7)
shoes = Table().with_columns('Standard Shoe', std_shoe, 'Carbon Fiber', carbon_fiber)
shoes

Standard Shoe,Carbon Fiber
54.2,52.1
51.5,50.3
55,54.8
53.8,51.2
52.1,51.9
56.4,53.5
50.9,49.1
52.7,52.4
54.5,52
53,51.1


### Hypothesis

### Data Checks

### Run Test

### Reporting Out

## <span style="color:blue;font-weight:bold;font-size:1.1em">Example 4: Nitrate Levels</span>

**Does the presence of a commercial agricultural runoff site significantly increase nitrate levels in a local river ecosystem?**

To address the impact of agricultural runoff, 12 unique geographical locations along a river basin were selected. At each location, water samples were collected at paired positions: exactly 100 meters upstream and 100 meters downstream from an agricultural drainage pipe. Nitrate concentration was measured in milligrams per liter (mg/L).

Test whether the values measured downstream were greater than those measure upstream at the $\alpha = 0.05$ level.

In [27]:
site = make_array('A-1', 'A-2', 'A-3', 'B-1', 'B-2', 'B-3', 'C-1', 'C-2', 'C-3', 'D-1', 'D-2', 'D-3')
upstream = make_array(4.2, 3.8, 5.0, 4.5, 4.1, 3.9, 5.5, 4.7, 4.2, 5.1, 4.6, 3.9)
downstream = make_array(5.1,4.9, 5.2, 6.0, 4.8, 4.3, 6.8, 5.9, 4.1, 6.3, 5.5, 4.7)
nitrate = Table().with_columns('Site', site, 'Upstream Level', upstream, 'Downstream Level', downstream)
nitrate

Site,Upstream Level,Downstream Level
A-1,4.2,5.1
A-2,3.8,4.9
A-3,5,5.2
B-1,4.5,6
B-2,4.1,4.8
B-3,3.9,4.3
C-1,5.5,6.8
C-2,4.7,5.9
C-3,4.2,4.1
D-1,5.1,6.3


### Hypothesis

### Data Checks

### Run Test

### Reporting Out